# Kayıp Fonksiyonları

Bu alıştırmada, Kayıp fonksiyonlarının `LinearRegression` modeli üzerindeki etkilerini karşılaştıracaksınız.

👇 Bu zorluk için kullanmak üzere bir CSV dosyası indirelim ve onu bir DataFrame'e dönüştürelim

In [25]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
561,0.69,735.0,294.0,220.5,3.5,0.40,15.850
537,0.86,588.0,294.0,147.0,7.0,0.40,31.525
299,0.86,588.0,294.0,147.0,7.0,0.25,31.695
201,0.86,588.0,294.0,147.0,7.0,0.10,28.565
312,0.74,686.0,245.0,220.5,3.5,0.25,13.810


🎯 Göreviniz, tasarımına göre bir seranın içindeki ortalama sıcaklığı tahmin etmektir. Sıcaklık tahminleriniz, her bir bitki için iklim ihtiyaçlarına göre uygun sera tasarımını seçmenize yardımcı olacaktır.

🌿 Bitkilerin küçük sıcaklık değişimlerini kaldırabildiğini, ancak sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geldiğini biliyorsunuz.

## 1. Teori

❓ Teorik olarak, bitkileri öldürme riskini sınırlamak için modelinizi hangi Kayıp fonksiyonu üzerinde eğitirsiniz?

<details>
<summary> 🆘 Cevap </summary>
    
Teorik olarak, Ortalama Kare Hata (MSE) Kayıp fonksiyonunu kullanırsınız. Bu, aykırı tahminleri cezalandırır ve modelinizin büyük hatalar yapmasını engeller. Bu, daha küçük sıcaklık değişimleri ve bitkiler için daha düşük risk sağlayacaktır.

</details>

> CEVABINIZI BURAYA YAZIN
> MSE

## 2. Uygulama

### 2.1 Ön İşleme

❓ Özellikleri standartlaştırın

In [26]:
from sklearn.preprocessing import StandardScaler

X = data.loc[:,'Relative Compactness':'Glazing Area']

# Fit scaler
scaler = StandardScaler().fit(X)

# Scale continuous features 
X_scaled = scaler.transform(X)


In [27]:
 X_scaled

array([[ 2.04177671, -1.78587489, -0.56195149, -1.47007664,  1.        ,
        -1.76044698],
       [ 2.04177671, -1.78587489, -0.56195149, -1.47007664,  1.        ,
        -1.76044698],
       [ 2.04177671, -1.78587489, -0.56195149, -1.47007664,  1.        ,
        -1.76044698],
       ...,
       [-1.36381225,  1.55394308,  1.12390297,  0.97251224, -1.        ,
         1.2440492 ],
       [-1.36381225,  1.55394308,  1.12390297,  0.97251224, -1.        ,
         1.2440492 ],
       [-1.36381225,  1.55394308,  1.12390297,  0.97251224, -1.        ,
         1.2440492 ]])

### 2.2 Modelleme

Bu bölümde, farklı Kayıp fonksiyonları üzerinde optimize edilmiş modelleri değerlendirerek teoriyi doğrulayacaksınız.

### En Küçük Kareler (MSE) Kaybı

❓ **En Küçük Kareler Kaybı** (MSE) üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

In [28]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate

model_sdg = SGDRegressor(loss="squared_error", max_iter=1000, random_state=42)

results = cross_validate(
    model_sdg,
    X_scaled,
    y,
    cv=10,
    scoring = ['r2','max_error']
)
results

{'fit_time': array([0.00739717, 0.00555134, 0.00648928, 0.00659275, 0.00552416,
        0.00518203, 0.00584769, 0.00543618, 0.00594068, 0.00572252]),
 'score_time': array([0.00254893, 0.00182176, 0.00505614, 0.00314546, 0.00145745,
        0.00194931, 0.00153375, 0.00170231, 0.0015502 , 0.00151658]),
 'test_r2': array([0.78749667, 0.90975959, 0.89479193, 0.88334616, 0.93122608,
        0.89671502, 0.92768161, 0.91637834, 0.89583701, 0.93904046]),
 'test_max_error': array([-9.78728135, -8.69148193, -8.7766776 , -9.19089935, -8.9242951 ,
        -8.62464292, -8.55781906, -8.80274655, -8.37697711, -7.66415004])}

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru ve bunu `r2` değişkeninde kaydedin
- Tüm katlarınızın °C cinsinden en büyük tek tahmin hatasını hesaplayın ve `max_error_celsius` değişkeninde kaydedin

(İpucu: `max_error` sklearn'de kabul edilen bir puanlama metriğidir)

In [29]:
r2 = results['test_r2'].mean()
r2

0.8982272875896993

In [34]:

max_error_celsius = abs(results['test_max_error']).max()
max_error_celsius

9.787281346962047

### Ortalama Mutlak Hata (MAE) Kaybı

Peki modelimizi MAE üzerinde optimize edersek ne olur?

❓ **MAE** Kaybı üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

<details>
<summary>💡 İpuçları</summary>

- MAE kaybı `SGDRegressor`'da doğrudan belirtilemez. Doğru parametreleri ayarlayarak tasarlanması gerekir

</details>

In [37]:
model_sdg = SGDRegressor(loss="epsilon_insensitive",epsilon = 0, max_iter=1000, random_state=42)

results_mae = cross_validate(
    model_sdg,
    X_scaled,
    y,
    cv=10,
    scoring = ['r2','max_error']
)
results_mae


{'fit_time': array([0.01321149, 0.00818729, 0.01004171, 0.00784183, 0.00842714,
        0.00785637, 0.00840044, 0.0092442 , 0.00790048, 0.00899863]),
 'score_time': array([0.00274706, 0.00174832, 0.00197339, 0.00170732, 0.00190854,
        0.00143576, 0.00174022, 0.00213814, 0.00157094, 0.00172257]),
 'test_r2': array([0.7432504 , 0.87564247, 0.8757397 , 0.84648879, 0.91754144,
        0.87354643, 0.9175109 , 0.8991184 , 0.87887026, 0.93556657]),
 'test_max_error': array([-11.06228474, -10.61762816, -10.62386046, -11.21001322,
        -11.13339995, -10.96225611, -10.81937682, -11.1413182 ,
        -10.92842515, -10.15643539])}

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru, bunu `r2_mae`'de saklayın
- Tüm katlarınızın en büyük tek tahmin hatasını, bunu `max_error_mae`'de saklayın

In [38]:
r2_mae = results_mae['test_r2'].mean()
r2_mae

0.8763275370982454

In [40]:
max_error_mae = abs(results_mae['test_max_error']).max()
max_error_mae

11.210013223184902

## 3. Sonuç

❓ Değerlendirdiğiniz modellerden hangisi göreviniz için en uygun görünüyor?

<details>
<summary> 🆘Cevap </summary>
    
İki model arasında ortalama çapraz doğrulanmış r2 skorları yaklaşık olarak benzer olmasına rağmen, MAE üzerinde optimize edilen modelin zaman zaman daha büyük hatalar yapma şansı daha fazladır, bu da bitkileri öldürme riskini artırır!
    
</details>

> CEVABINIZI BURAYA YAZIN

model_sdg

# 🏁 Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [41]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/mert/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/mert/Sprint-16/S16D4-S-loss-functions/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 3 items

test_loss_functions.py::TestLossFunctions::test_max_error_order PASSED   [ 33%]
test_loss_functions.py::TestLossFunctions::test_r2 PASSED                [ 66%]
test_loss_functions.py::TestLossFunctions::test_r2_mae PASSED            [100%]

============================== 3 passed in 0.29s ===============================


💯 You can commit your code:

git add tests/loss_functions.pickle

git commit -m 'Completed loss_functions step'

git push origin master

